In [1]:
# ============================================================
# GEMINI LLM EVALUATION
# Accuracy + Precision + Recall + F1 Score
# Google Colab - Single Cell
# ============================================================

# Install libraries
!pip install -q -U google-genai scikit-learn

# ------------------------------------------------------------
# 1. Imports
# ------------------------------------------------------------

from google import genai
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)
import getpass


In [3]:
# ------------------------------------------------------------
# STEP 3: Enter Gemini API Key
# ------------------------------------------------------------
import os
from google import genai
from google.colab import userdata

#API_KEY = put the new api key here created latest
api_key = userdata.get("GOOGLE_API_KEY")
#client = genai.Client(api_key=api_key)
os.environ["GOOGLE_API_KEY"] = api_key
#api_key = getpass.getpass("Enter your Gemini API Key: ")

client = genai.Client(api_key=api_key)

In [4]:
# ------------------------------------------------------------
# 3. Test Dataset
#
# 1 = POSITIVE
# 0 = NEGATIVE
# ------------------------------------------------------------

reviews = [
    "This movie was fantastic and I loved it.",
    "The movie was terrible and boring.",
    "Excellent acting and a wonderful story.",
    "I hated this movie.",
    "This was one of the best movies I have watched.",
    "The film was disappointing and poorly made.",
    "Amazing movie with brilliant performances.",
    "The story was dull and uninteresting.",
    "I really enjoyed watching this film.",
    "This was a complete waste of time."
]

# Ground-truth / correct answers
true_labels = [
    1, 0, 1, 0, 1,
    0, 1, 0, 1, 0
]


# ------------------------------------------------------------
# 4. Ask Gemini to classify each review
# ------------------------------------------------------------

predicted_labels = []

for review in reviews:

    prompt = f"""
Classify the following movie review.

Return ONLY one word:

POSITIVE
or
NEGATIVE

Review:
{review}
"""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    answer = response.text.strip().upper()

    # Convert Gemini answer to numeric label
    if "POSITIVE" in answer:
        prediction = 1
    else:
        prediction = 0

    predicted_labels.append(prediction)

    print("\nReview:", review)
    print("Gemini:", answer)


# ------------------------------------------------------------
# 5. Calculate evaluation metrics
# ------------------------------------------------------------

accuracy = accuracy_score(
    true_labels,
    predicted_labels
)

precision = precision_score(
    true_labels,
    predicted_labels,
    zero_division=0
)

recall = recall_score(
    true_labels,
    predicted_labels,
    zero_division=0
)

f1 = f1_score(
    true_labels,
    predicted_labels,
    zero_division=0
)


# ------------------------------------------------------------
# 6. Display results
# ------------------------------------------------------------

print("\n======================================")
print("       GEMINI LLM EVALUATION")
print("======================================")

print("Accuracy  :", round(accuracy, 4))
print("Precision :", round(precision, 4))
print("Recall    :", round(recall, 4))
print("F1 Score  :", round(f1, 4))

print("\nPercentage:")
print("Accuracy  :", round(accuracy * 100, 2), "%")
print("Precision :", round(precision * 100, 2), "%")
print("Recall    :", round(recall * 100, 2), "%")
print("F1 Score  :", round(f1 * 100, 2), "%")


# ------------------------------------------------------------
# 7. Confusion Matrix
# ------------------------------------------------------------

cm = confusion_matrix(true_labels, predicted_labels)

print("\nConfusion Matrix:")
print(cm)

TN, FP, FN, TP = cm.ravel()

print("\nTrue Positive  (TP):", TP)
print("True Negative  (TN):", TN)
print("False Positive (FP):", FP)
print("False Negative (FN):", FN)


Review: This movie was fantastic and I loved it.
Gemini: POSITIVE

Review: The movie was terrible and boring.
Gemini: NEGATIVE

Review: Excellent acting and a wonderful story.
Gemini: POSITIVE

Review: I hated this movie.
Gemini: NEGATIVE

Review: This was one of the best movies I have watched.
Gemini: POSITIVE

Review: The film was disappointing and poorly made.
Gemini: NEGATIVE

Review: Amazing movie with brilliant performances.
Gemini: POSITIVE

Review: The story was dull and uninteresting.
Gemini: NEGATIVE

Review: I really enjoyed watching this film.
Gemini: POSITIVE

Review: This was a complete waste of time.
Gemini: NEGATIVE

       GEMINI LLM EVALUATION
Accuracy  : 1.0
Precision : 1.0
Recall    : 1.0
F1 Score  : 1.0

Percentage:
Accuracy  : 100.0 %
Precision : 100.0 %
Recall    : 100.0 %
F1 Score  : 100.0 %

Confusion Matrix:
[[5 0]
 [0 5]]

True Positive  (TP): 5
True Negative  (TN): 5
False Positive (FP): 0
False Negative (FN): 0


In [6]:
# ------------------------------------------------------------
# 2. Dataset
#
# 1 = POSITIVE
# 0 = NEGATIVE
#
# Some sentences are intentionally tricky.
# ------------------------------------------------------------

reviews = [
    "The movie was absolutely wonderful.",
    "I completely hated this movie.",
    "The acting was good but the story was terrible.",
    "Not the worst movie I have ever seen.",
    "I expected it to be terrible, but it surprised me.",
    "The visuals were beautiful, unfortunately everything else was awful.",
    "I wouldn't say the movie was bad.",
    "Great, another two hours of my life wasted.",
    "The movie could have been better, but I enjoyed it.",
    "Amazing acting couldn't save this boring movie."
]


# Human ground-truth labels
true_labels = [
    1,   # wonderful
    0,   # hated
    0,   # overall negative
    1,   # not the worst
    1,   # pleasantly surprised
    0,   # overall negative
    1,   # not bad
    0,   # sarcasm - negative
    1,   # enjoyed
    0    # boring movie
]


# ------------------------------------------------------------
# 3. Ask Gemini
# ------------------------------------------------------------

gemini_predictions = []

print("\n========== GEMINI PREDICTIONS ==========\n")

for i, review in enumerate(reviews):

    prompt = f"""
Classify the overall sentiment of this movie review.

Return ONLY:

POSITIVE

or

NEGATIVE

Review:
{review}
"""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    answer = response.text.strip().upper()

    if "POSITIVE" in answer:
        prediction = 1
    else:
        prediction = 0

    gemini_predictions.append(prediction)

    print(i + 1, review)
    print("Human :", "POSITIVE" if true_labels[i] == 1 else "NEGATIVE")
    print("Gemini:", "POSITIVE" if prediction == 1 else "NEGATIVE")
    print()


# ------------------------------------------------------------
# 4. Calculate REAL Gemini performance
# ------------------------------------------------------------

accuracy = accuracy_score(true_labels, gemini_predictions)

precision = precision_score(
    true_labels,
    gemini_predictions,
    zero_division=0
)

recall = recall_score(
    true_labels,
    gemini_predictions,
    zero_division=0
)

f1 = f1_score(
    true_labels,
    gemini_predictions,
    zero_division=0
)


# ------------------------------------------------------------
# 5. Results
# ------------------------------------------------------------

print("\n======================================")
print("       GEMINI EVALUATION RESULTS")
print("======================================")

print("Accuracy  :", round(accuracy, 3))
print("Precision :", round(precision, 3))
print("Recall    :", round(recall, 3))
print("F1 Score  :", round(f1, 3))


# ------------------------------------------------------------
# 6. Confusion Matrix
# ------------------------------------------------------------

cm = confusion_matrix(true_labels, gemini_predictions)

print("\nConfusion Matrix:")
print(cm)

TN, FP, FN, TP = cm.ravel()

print("\nTP =", TP)
print("TN =", TN)
print("FP =", FP)
print("FN =", FN)


# ------------------------------------------------------------
# 7. Show formulas manually
# ------------------------------------------------------------

print("\n======================================")
print("        MANUAL CALCULATION")
print("======================================")

print("\nAccuracy")
print("(TP + TN) / (TP + TN + FP + FN)")
print("=", (TP + TN), "/", (TP + TN + FP + FN))
print("=", round(accuracy, 3))

print("\nPrecision")
print("TP / (TP + FP)")
print("=", TP, "/", (TP + FP))
print("=", round(precision, 3))

print("\nRecall")
print("TP / (TP + FN)")
print("=", TP, "/", (TP + FN))
print("=", round(recall, 3))

print("\nF1 Score")
print("2 * Precision * Recall / (Precision + Recall)")
print("=", round(f1, 3))


========== GEMINI PREDICTIONS ==========

1 The movie was absolutely wonderful.
Human : POSITIVE
Gemini: POSITIVE

2 I completely hated this movie.
Human : NEGATIVE
Gemini: NEGATIVE

3 The acting was good but the story was terrible.
Human : NEGATIVE
Gemini: NEGATIVE

4 Not the worst movie I have ever seen.
Human : POSITIVE
Gemini: POSITIVE

5 I expected it to be terrible, but it surprised me.
Human : POSITIVE
Gemini: POSITIVE

6 The visuals were beautiful, unfortunately everything else was awful.
Human : NEGATIVE
Gemini: NEGATIVE

7 I wouldn't say the movie was bad.
Human : POSITIVE
Gemini: POSITIVE

8 Great, another two hours of my life wasted.
Human : NEGATIVE
Gemini: NEGATIVE

9 The movie could have been better, but I enjoyed it.
Human : POSITIVE
Gemini: POSITIVE

10 Amazing acting couldn't save this boring movie.
Human : NEGATIVE
Gemini: NEGATIVE


       GEMINI EVALUATION RESULTS
Accuracy  : 1.0
Precision : 1.0
Recall    : 1.0
F1 Score  : 1.0

Confusion Matrix:
[[5 0]
 [0 5]]

TP